# LEVELS Benchmark — Prompt-Variant Sweep

Every Level 1 and Level 2 task above also has 5 hand-authored alternate phrasings
(`task.prompt_variants`) of the *exact same underlying ask* — the ground truth, expected
tools, and scoring rubric are identical across styles; only the wording/register changes:

- **novice** — plain-language, non-technical phrasing a first-time user might type
- **expert** — technical/jargon-heavy phrasing using DSI field names and operation terms
- **terse** — minimal, keyword-only phrasing
- **verbose** — over-explained, context-heavy phrasing with extra framing
- **casual** — informal, lowercase, abbreviation-heavy phrasing

`expand_prompt_variants()` turns each base task into 6 runs (`standard` + the 5 styles
above), and `run_prompt_variant_sweep()` executes every one of those **in full isolation**
(a fresh `DSIExplorer` per run, regardless of level) so a later-run style never benefits
from conversational state left over by an earlier-run style of the *same* task — otherwise
a model could "pass" a harder phrasing for free just because an earlier easier phrasing
already loaded the right database in a shared session.

**Cost/time note:** this expands 13 Level 1 + 11 Level 2 = 24 base tasks × 6 phrasings ×
`len(MODELS_TO_RUN)` models into **144 × n_models agent runs** — substantially more calls
than any other section in this notebook. Consider trying it on a small subset first (e.g.
`PROMPT_VARIANT_BASE_TASKS[:3]`) before running the full sweep, and/or temporarily reducing
`MODELS_TO_RUN` to one model.

Shared harness code (scoring, parallel runner, reports) lives in `levels_bench/`, and the task
definitions in `levels_bench/tasks/`. Each (model, task) pair runs as an independent job in a
pool of worker processes, so `max_workers` runs proceed at once. Every finished result is
appended to `levels_benchmark_results/<run_id>_<label>.partial.jsonl` as it completes. If a run
is interrupted, call `run_benchmark(..., resume=True)` to run only what's missing.

## Setup

In [ ]:
# Run once per environment (not needed on every run):
# %pip install -q -r requirements.txt
# %pip install -q --no-build-isolation -e ../..

In [ ]:
from levels_bench import expand_prompt_variants, init, run_benchmark, flatten, results_to_dataframe, load_results, reports, debug_task
from levels_bench.tasks import LEVEL1_TASKS, LEVEL2_TASKS, get_task

cfg = init(
    # models_to_run=["aws-gov.gpt-5.4"],   # default: every model in levels_bench.config.POSSIBLE_MODELS
    # judge_models=[...],                   # default: same list; judged scores are averaged across them
    max_workers=4,                          # concurrent agent runs; raise if the portal's rate limit allows
    task_timeout_sec=180,                   # per agent turn
    # skip_network=True,                    # skip tasks that download from oceans11.lanl.gov
)

In [ ]:
PROMPT_VARIANT_BASE_TASKS = list(LEVEL2_TASKS)   # add LEVEL1_TASKS for the full sweep; try [:3] first
pv_tasks = expand_prompt_variants(PROMPT_VARIANT_BASE_TASKS)
print(f"{len(PROMPT_VARIANT_BASE_TASKS)} base tasks -> {len(pv_tasks)} phrasings x "
      f"{len(cfg.models_to_run)} models = {len(pv_tasks) * len(cfg.models_to_run)} runs")

## Run

In [ ]:
pv_results = flatten(run_benchmark(pv_tasks, "prompt_variants"))
# pv_results = load_results("prompt_variants")
pv_df = results_to_dataframe(pv_results)
pv_df

## Results

In [ ]:
reports.prompt_styles(pv_df)

In [ ]:
reports.failures(pv_df)

In [ ]:
reports.timing_breakdown(pv_results)

## Inspect / debug

Transcripts are recorded for every run. `debug_task` reruns one task in this process with live output.

In [ ]:
# reports.transcript(pv_results, "L2_physics_contact_report__terse")
# tr = debug_task(get_task("L2_physics_contact_report__terse"), model="anthropic.claude-sonnet-4-5-20250929-v1:0")